# Task 3 — Loan Default & Expected Loss

JPMorgan Chase Quantitative Research — Forage Simulation

Build a model to predict the **probability of default (PD)** for a borrower, then use it to calculate the **expected loss** on a loan (assuming a **10% recovery rate**).

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))

import matplotlib.pyplot as plt
import pandas as pd

from loan_risk_model import (
    FEATURE_COLUMNS,
    RECOVERY_RATE,
    calculate_expected_loss,
    compare_models,
    get_model,
    get_probability_of_default,
    load_data,
)

plt.style.use("seaborn-v0_8-whitegrid")
DATA_PATH = Path("Task 3 and 4 Loan Data.csv")
print("Setup complete.")

## 1. Load and Explore the Data

In [ ]:
df = load_data()

print(f"Borrowers:     {len(df):,}")
print(f"Default rate:  {df['default'].mean():.1%}")
print(f"Features:      {', '.join(FEATURE_COLUMNS)}")
df.head()

## 2. Exploratory Analysis

Understand which borrower characteristics relate to default risk.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df["default"].value_counts().plot(
    kind="bar", ax=axes[0], color=["#2e75b6", "#c55a11"], rot=0
)
axes[0].set_title("Default vs Non-Default", fontweight="bold")
axes[0].set_xlabel("Default")
axes[0].set_ylabel("Count")
axes[0].set_xticklabels(["No Default", "Default"])

default_by_fico = df.groupby("default")["fico_score"].mean()
axes[1].bar(["No Default", "Default"], default_by_fico.values, color=["#2e75b6", "#c55a11"])
axes[1].set_title("Average FICO Score by Default Status", fontweight="bold")
axes[1].set_ylabel("FICO Score")

plt.tight_layout()
plt.show()

corr = df[FEATURE_COLUMNS + ["default"]].corr()["default"].drop("default").sort_values()
print("Correlation with default:")
print(corr)

## 3. Compare Prediction Models

We compare three approaches for predicting probability of default:
- **Logistic Regression** — simple, interpretable baseline
- **Decision Tree** — captures non-linear rules
- **Random Forest** — ensemble of trees, often strongest performance

In [ ]:
comparison = compare_models(df)
comparison

## 4. Train the Final Model

We use **Logistic Regression** (highest ROC AUC in our comparison) as the production model.

In [ ]:
model = get_model()
print("Model trained on full dataset.")
print(f"Recovery rate assumed: {RECOVERY_RATE:.0%}")
print(f"Loss given default:    {1 - RECOVERY_RATE:.0%}")

## 5. Expected Loss Formula

Expected loss combines the probability of default with exposure and recovery:

$$\text{Expected Loss} = PD \times \text{Loan Amount Outstanding} \times (1 - \text{Recovery Rate})$$

With a **10% recovery rate**, we assume **90%** of the outstanding loan is lost if the borrower defaults.

## 6. Expected Loss Function

This is the deliverable: pass loan details, get expected loss.

In [ ]:
def calculate_loan_expected_loss(loan) -> float:
    """
    Calculate expected loss for a loan.

    Parameters
    ----------
    loan : dict or Series
        Borrower/loan features including loan_amt_outstanding.

    Returns
    -------
    float
        Expected loss in dollars.
    """
    return calculate_expected_loss(loan)


def get_loan_pd(loan) -> float:
    """Return predicted probability of default for a loan."""
    return get_probability_of_default(loan)

## 7. Sample Test Cases

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from loan_risk_model import calculate_expected_loss, get_probability_of_default

samples = [
    {
        "label": "Low-risk borrower",
        "loan": {
            "credit_lines_outstanding": 0,
            "loan_amt_outstanding": 3000,
            "total_debt_outstanding": 2500,
            "income": 85000,
            "years_employed": 6,
            "fico_score": 720,
        },
    },
    {
        "label": "High-risk borrower",
        "loan": {
            "credit_lines_outstanding": 5,
            "loan_amt_outstanding": 8000,
            "total_debt_outstanding": 18000,
            "income": 35000,
            "years_employed": 1,
            "fico_score": 480,
        },
    },
    {
        "label": "Actual default from dataset",
        "loan": df[df["default"] == 1].iloc[0][FEATURE_COLUMNS].to_dict(),
    },
]

print(f"{'Profile':<28} {'PD':>8} {'Expected Loss':>16}")
print("-" * 54)
for sample in samples:
    pd_val = get_probability_of_default(sample["loan"])
    el = calculate_expected_loss(sample["loan"])
    print(f"{sample['label']:<28} {pd_val:>7.1%} ${el:>14,.2f}")

## 8. Try Your Own Loan

Enter borrower details to get PD and expected loss.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from loan_risk_model import calculate_expected_loss, get_probability_of_default

loan = {
    "credit_lines_outstanding": int(input("Credit lines outstanding: ")),
    "loan_amt_outstanding": float(input("Loan amount outstanding ($): ")),
    "total_debt_outstanding": float(input("Total debt outstanding ($): ")),
    "income": float(input("Annual income ($): ")),
    "years_employed": int(input("Years employed: ")),
    "fico_score": int(input("FICO score: ")),
}

pd_val = get_probability_of_default(loan)
el = calculate_expected_loss(loan)

print(f"\nProbability of default: {pd_val:.2%}")
print(f"Expected loss:          ${el:,.2f}")